In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals

print("Ready!")

## *PART II: The Philippines as a Biodiversity Hotspot*

### Vegetation types & forest cover

Yesterday and this morning, we classified climate zones using [Köppen's system](https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification) — and Köppen, originally a botanist, designed his thresholds specifically so that climate zones would line up with **vegetation zones** (Af regions look like rainforest, Aw regions look like savanna, and so on). In the next part, we look at *actual* vegetation data instead of using climate as a proxy for it.

We'll **zoom into Negros** and use the **[Hansen Global Forest Change dataset](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/download.html)** (Hansen et al., 2013, *Science* — 👉 https://doi.org/10.1126/science.1244693), which maps global tree cover and forest loss from satellite imagery at 30 m resolution. You can browse it interactively yourself at 👉 https://glad.earthengine.app/view/global-forest-change.

For now we only need the `treecover2000` variable (percent canopy cover in the year 2000) from `negros_forest_change.nc`. The file is under `../data/processed/day2/`

In [ ]:
# --- Load Negros forest cover data ---
forest_ds = xr.open_dataset("../data/processed/day2/negros_forest_change.nc")
forest_ds

**Inspect the data** with a 1x2 (one row, two columns) subplot. Left, you plot `treecover2000`, right, you plot `lossyear`:

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,5))                           # ✏️ initiate 1x2 subplot with figsize=(10,5)
forest_ds["treecover2000"].plot(ax=axes[0], cmap="Greens")              # ✏️ plot the treecover2000 variable from forest_ds, use cmap="Greens" as colormap
forest_ds["lossyear"].plot(ax=axes[1], cmap="Reds", vmin=1, vmax=23)    # ✏️ plot the lossyear variable from forest_ds, use cmap="Reds" as colormap

# don't bother with titles or layout, it's just a quick data inspection

> **Discussion:** What do you think the data represents? What is `treecover2000`, and what is `lossyear`? *Hint: the unit is 'years since 2000'*. You might also have noticed that the figure above took a while to plot. Why is that, you think?

The file has a very high resolution of 30m. How many data points are you plotting in the point above? **✏️ Calculate it below!**

In [ ]:
# Hint:
# number of data points = two variables x number of latitudes x number of longitudes
nr = 2 * len(forest_ds.latitude) * len(forest_ds.longitude)
nr

💡 If you got that right, you should see that, above, you are plotting *42 240 000 datapoints*, that's a lot. To make our lives a bit easier, let's reduce that **by coarsening it** a bit so maps stay fast to draw:

In [ ]:
treecover = forest_ds["treecover2000"]
treecover_coarse = treecover.coarsen(latitude=10, longitude=10, boundary="trim").mean()     # ✏️ what do you think specifying latitude=10, longitude=10 does?
treecover_coarse

**🖊️ How many datapoints does treecover contain?**

In [ ]:
# Hint: same as before, but only for treecover_coarse
nr = len(treecover_coarse.latitude) * len(treecover_coarse.longitude)
nr

*211 200 datapoints*, good, that's a lot less than before. Let's see if that plots a bit faster:

**🖊️ Plot the tree cover map and mark Bacolod.**

In [ ]:
#  visualize the tree cover data for Negros and mark Bacolod:
fig, ax = plt.subplots(figsize=(7, 7))                                  # ✏️ initialize fig, ax
treecover_coarse.plot(ax=ax, cmap="Greens", vmin=0, vmax=100,           # ✏️ plot treecover_coarse
                       cbar_kwargs={"label": "Tree cover 2000 (%)"})


# ✏️ use ax.scatter(...) and ax.text(...) to add a marker and label for Bacolod
# coordinates: 122.95, 10.68
ax.scatter(122.95, 10.68, color="black", s=60, zorder=5)                
ax.text(122.97, 10.69, "Bacolod", fontsize=9)
ax.set_title("Tree Cover — Negros (year 2000)")
ax.set_aspect("equal")

> **Discussion:** Where on the map is Negros most forested? Does that line up with the wetter (Af/Am) vs. drier (Aw) zones from Part I's homemade climate map?

**🖊️ Summary statistics:** For the entire figure, we can compute the **mean tree cover percentage** simply by using the `.mean()` method. Try it:

In [ ]:
mean_cover = float(treecover.mean())            # ✏️ use .mean() on the treecover variable
print(mean_cover)

💡 How could you compute the percentage of **pixels with more than 50% tree cover**?

It's super easy, but probably not intuitive for you just yet. Let's guide you through this with an example. Say we take some 2x2 array:

In [ ]:
some_array = np.array([[5, 3], [2, 8]])
some_array

**Where is it > 4?** You can see: first row, first column (number 5); and second row, second column (number 8). **See what this does:**

In [ ]:
some_array > 4

So that gives `True` where the values in some_array are > 4, and `False` where they are not.

In programming, `True` and `False` are often **synonymous** with `1` and `0`, and in this case, we can actually use it just like that, look:

In [ ]:
(some_array > 4).mean()

Two times True (1) and two times False (0) -> the mean is 0.5. So on average, 50% of the values in some_array are > 4. **All we have to do to get a percentage is multiply by 100**:

In [ ]:
my_percentage = (some_array > 4).mean() * 100

print(f"Hey look, {my_percentage}% of the values are greater than 4!")

**🖊️ Your turn: compute what percentage of Negros is forest.** Let's say a pixel with more than 50% tree cover is forest. From the `treecover` DataArray, compute the percentage of the image that meets this criterion:

In [ ]:
percentage_forest = (treecover > 50).mean() * 100           # ✏️ percentage of pixels with treecover > 50%

print(f"Pixels with >50% tree cover: {float(percentage_forest)}%")

> 💬 **Discussion:** Is this what you expected? A mean tree cover of about 12.5% and a mean forest cover of about 14%? It might be quite a bit lower than what you thought. Why? *(Hint: what area are we taking the mean over?)*

Let's take it up one notch and only compute forest cover over land by using the **land sea mask**. The file is called `land-sea-mask_0p00833333.nc` and is under `../data/processed/`:

In [ ]:
lsm = xr.open_dataarray("../data/processed/land-sea-mask_0p00833333.nc")    # ✏️ open with xr.open_dataarray()
lsm.plot()

At 1km resolution, the `lsm` DataArray for the Philippines shows where there's land (`True`, or `1`) and where there's ocean (`False` or `0`)&mdash;it's a **binary** mask.

Even though its **resolution and coordinates are not the same as for** `treecover`, `xarray` is able to compare the two without any problem and allows you to play with the data. Look:

In [ ]:
treecover.where(lsm==1).plot()

---

### Satellite imagery

What does a satellite actually "see"? Not a photograph — a grid of numbers, one per pixel, for each of several **spectral bands** (wavelength ranges). Some of those bands aren't even visible to our eyes.

We'll use the **[Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2)** mission (ESA/Copernicus): two satellites launched in 2015 and 2017 that jointly revisit every point on Earth every ~5 days, at 10-60 m resolution. `negros_sentinel2.nc` has two cloud-free mosaics over Negros — 2018 ("old") and 2024 ("new") — with these bands:
- `red`, `nir` (near-infrared) — raw reflectance values. That is: the satellite beams red and near-infrared to the Earth and measures how much reflects back.
- `visual_r`, `visual_g`, `visual_b` — a ready-made true-colour image (what your eyes would actually see)

*(Why 2018 and not, say, 2000? Sentinel-2 only launched in 2015 — before that, this specific satellite simply didn't exist. 2018 is close to the earliest point with reliably cloud-free coverage over Negros.)*

A single band is just a 2D array of numbers — let's look at one directly:

In [ ]:
# --- Load Sentinel-2 data ---
s2_ds = xr.open_dataset("../data/processed/day2/negros_sentinel2.nc")
print(s2_ds.time.values)   # the two available dates

# =============================================================
# 💡 EXAMPLE — a single band is just numbers
# =============================================================
red_new = s2_ds["red"].sel(time="2024-04-03")

fig, ax = plt.subplots(figsize=(6, 6))
red_new.plot(ax=ax, cmap="gray")
ax.set_title("Red band only — just one number per pixel")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> **Discussion:** What do you notice about the red band? What type of surface reflects a lot of red light?

**🖊️ Data quality check:** what percentage of values in `red_new` are NaNs?

Hints: remember xarray's `shape`, `np.isnan` and `np.sum`

In [ ]:
# ✏️ Compute the percentage of pixels that has a missing value
# Hint: number of NaNs / total number of pixels * 100
np.sum(np.isnan(red_new)) / np.prod(red_new.shape) * 100

To get an actual photo-like image, we need to combine three bands (red, green, blue) into a single array of shape `(rows, cols, 3)` — one extra dimension holding the 3 colour channels for every pixel. `np.dstack` stacks 2D arrays along a new third axis to do exactly this:

In [ ]:
# =============================================================
# 💡 EXAMPLE — stacking 3 bands into a true-colour image
# =============================================================
s2_new = s2_ds.sel(time="2024-04-03")

# 💡 a few pixels are NaN - fill with 0 (black) so imshow doesn't complain
r = np.nan_to_num(s2_new["visual_r"].values)
g = np.nan_to_num(s2_new["visual_g"].values)
b = np.nan_to_num(s2_new["visual_b"].values)

rgb_new = np.dstack([r, g, b]).astype(np.uint8)   # shape (rows, cols, 3)
print("rgb_new shape:", rgb_new.shape)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb_new)          # 💡 imshow displays a (rows, cols, 3) array directly as a photo
ax.set_title("True-colour image — Negros, 2024")
ax.axis("off")
plt.tight_layout()
plt.show()

### 🖊️ Your turn: old vs. new, side by side

Build the same kind of true-colour image for the **2018** ("old") time slice, and plot it next to the 2024 image you already have — a two-panel figure, same subplot skills as yesterday.

**Tips:**
- The "old" date is `"2018-02-09"`
- Reuse `rgb_new` (already built above) for the right-hand panel

In [ ]:
# --- Build the "old" (2018) true-colour image ---
s2_old = s2_ds.sel(time="2018-02-09")          # ✏️ pick the 2018 date, same as s2_new above but for "2018-02-09"

r_old = np.nan_to_num(s2_old["visual_r"].values)          # ✏️ same recipe as r, g, b above — just swap s2_new for s2_old
g_old = np.nan_to_num(s2_old["visual_g"].values)
b_old = np.nan_to_num(s2_old["visual_b"].values)
rgb_old = np.dstack([r_old, g_old, b_old]).astype(np.uint8)

# --- Two-panel comparison (same subplot pattern as Day 1) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].imshow(rgb_old)
axes[0].set_title("2018")
axes[1].imshow(rgb_new)
axes[1].set_title("2024")

for ax in axes:
    ax.axis("off")

fig.suptitle("Negros, Sentinel-2 true colour — 2018 vs. 2024")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** What visible changes can you spot between 2018 and 2024, just by eye? Can you point out forest, farmland, urban areas, and coastline in the images — and does what you see line up with what you'd expect from the `treecover2000` map, and from Bacolod's known location?

---

### Fauna

Climate and vegetation set the stage — now let's look at who actually lives there. We'll use occurrence records from **[GBIF](https://www.gbif.org/)** (the Global Biodiversity Information Facility), a free, public database aggregating millions of species sightings and museum specimens worldwide from thousands of institutions.

Two flagship species first:
- **[Philippine Eagle](https://en.wikipedia.org/wiki/Philippine_eagle)** (*Pithecophaga jefferyi*) — one of the world's largest and rarest eagles, endemic to the Philippines, Critically Endangered, and entirely forest-dependent. 👉 https://www.gbif.org/species/2480381
- **[Large flying fox](https://en.wikipedia.org/wiki/Large_flying_fox)** (*Pteropus vampyrus*) — one of the largest bats in the world, found across the Philippine archipelago. 👉 https://www.gbif.org/species/5218644

In [ ]:
# --- Load occurrence records ---
eagle = pd.read_csv("../data/processed/day2/gbif_philippine_eagle.csv")
flying_fox = pd.read_csv("../data/processed/day2/gbif_flying_fox.csv")

print(f"Philippine Eagle records: {len(eagle)}")
print(f"Flying fox records: {len(flying_fox)}")
eagle.head()

### 🖊️ Your turn: map the occurrence points

No basemap needed — a plain `ax.scatter(lon, lat, ...)` is enough, and we can reuse yesterday's `land_sea_mask.nc` contour lines as a rough Philippines outline, exactly like you did for the temperature/precipitation maps.

Make a two-panel figure: eagle records on the left, flying fox records on the right.

In [ ]:
lsm_ph = xr.open_dataarray("../data/processed/land-sea-mask_0p00833333.nc")

fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].scatter(eagle["lon"], eagle["lat"], color="firebrick", s=20, alpha=0.7)     # ✏️ eagle["lon"], eagle["lat"]
axes[0].set_title(f"Philippine Eagle ({len(eagle)} records)")

axes[1].scatter(flying_fox["lon"], flying_fox["lat"], color="steelblue", s=20, alpha=0.7)     # ✏️ flying_fox["lon"], flying_fox["lat"]
axes[1].set_title(f"Flying Fox ({len(flying_fox)} records)")

for ax in axes:
    ax.contour(lsm_ph.lon, lsm_ph.lat, lsm_ph.values,
               levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** What patterns do you notice in the observation records of the Philippine Eagle and flying fox? Given that the eagle is entirely forest-dependent and Critically Endangered, what does that clustering tell you? *(Keep this in mind — we'll come back to forest loss and its effect on species like this in Part IV.)*

---

Two species is a start, but let's look at a much richer picture: **every terrestrial mammal record for the Philippines** in GBIF.

In [ ]:
# --- Load the full mammal community dataset ---
mammals = pd.read_csv("../data/processed/day2/gbif_mammals_philippines.csv")
print(f"{len(mammals)} records, {mammals['species'].nunique()} species")
mammals.head()

### 🖊️ Your turn: which mammal groups dominate?

Use `.value_counts()` on the `order` column to see which taxonomic groups have the most records. There's a surprise waiting.

In [ ]:
order_counts = mammals["order"].value_counts()     # ✏️ mammals["order"].value_counts()
print(order_counts)

> 💬 **Discussion:** Which order dominates, and by how much? Does that match what you'd naively guess about "Philippine mammals"? *(Hint: think about how easy each group is to survey. Record counts reflect sampling effort as much as true abundance!)*

---

### Data quality check

Before we filter and plot, let's check how complete the columns we're about to rely on actually are — real-world biodiversity data is rarely 100% populated, and it's good practice to check *before* you filter on a column, not after.

In [ ]:
# =============================================================
# 💡 EXAMPLE — checking how many values are missing in a column, here "county"
# =============================================================
print(mammals["county"].isnull().sum(), "missing out of", len(mammals))          # count of missing values
print(f"{mammals['county'].isnull().mean()*100:.1f}% missing")                                        # 💡 same trick as before: True=1/False=0, so .mean() = fraction missing

**🖊️ check null rates** on the columns we're about to use

We're about to filter records to Negros using `stateProvince`, and later make a bar chart of `iucnRedListCategory`. Check the missing-value fraction for both (plus `family` and `order`, for comparison, since we know those are complete).

In [ ]:
columns_to_check = ["stateProvince", "iucnRedListCategory", "family", "order"]

for col in columns_to_check:
    frac_missing = mammals[col].isnull().mean()     # ✏️ mammals[col].isnull().mean()
    print(f"{col:20s}: {frac_missing * 100:5.1f}% missing")

> 💬 **Discussion:** `stateProvince` and `iucnRedListCategory` both have a meaningful chunk of missing values, while `family`/`order` are essentially complete. What does that mean for the filtering and bar chart we're about to make? *(We'll simply be working with a subset of all records — not a bias-free sample, but still informative.)*

---

### Filtering to Negros

`stateProvince` isn't perfectly clean text — a look at the raw values shows entries like `"Negros Oriental"`, `"Negros Occidental"`, `"Negros I"`, `"Negros Id."`. Rather than matching one exact string, we can check whether `"Negros"` appears *anywhere* in the text using `.str.contains()`:

In [ ]:
# =============================================================
# 💡 EXAMPLE — filtering rows by a partial text match
# =============================================================
sample_values = pd.Series(["Negros Oriental", "Negros Occidental", "Cebu", "Negros I", np.nan])

is_negros = sample_values.str.contains("Negros", case=False, na=False)   # 💡 na=False: treat missing values as "no match", not an error
print(is_negros)

### 🖊️ Your turn: filter the mammal records to Negros, and map them

**Tips:**
- Same pattern as the example, applied to `mammals["stateProvince"]`
- Use `mammals[is_negros]` to keep only the matching rows (same boolean-filtering idea as yesterday's `df[condition]`)
- Reuse the scatter + `land_sea_mask.nc` contour pattern from the eagle/flying fox map — but this time, zoom the axes to Negros using `ax.set_xlim(122.3, 123.4)` and `ax.set_ylim(9.8, 11.0)` (the same bounding box as the Sentinel-2/Hansen data)

In [ ]:
is_negros = mammals["stateProvince"].str.contains("Negros", case=False, na=False)          # ✏️ same pattern as sample_values above, applied to the real column
negros_mammals = mammals[is_negros]               # ✏️ same boolean-filtering idea as df[condition] from Day 1

fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(negros_mammals["decimalLongitude"], negros_mammals["decimalLatitude"],
           color="darkorange", s=10, alpha=0.5)
ax.contour(lsm_ph.lon, lsm_ph.lat, lsm_ph.values,
           levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7)

ax.set_xlim(122.3, 123.4)          # ✏️ zoom to Negros
ax.set_ylim(9.8, 11.0)
ax.set_title(f"Mammal occurrence records — Negros (total: {len(negros_mammals)})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

> 💬 **Discussion:** Notice the points scattered far outside Negros itself (even outside the Philippines!) if you check `negros_mammals["decimalLongitude"].describe()` — a good reminder that a `stateProvince` text match isn't a perfect filter; some records are simply mislabeled or mis-geocoded. Real biodiversity data always needs this kind of scrutiny.

### 🖊️ Conservation status bar chart

Let's check how many animals belong to the different categories! We'll make a bar chart of `iucnRedListCategory` value counts for the Negros mammal subset.

**Tip:**
- `.value_counts()` on `negros_mammals["iucnRedListCategory"]` gives you counts per category directly

In [ ]:
iucn_counts = negros_mammals["iucnRedListCategory"].value_counts()    # ✏️ negros_mammals["iucnRedListCategory"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(iucn_counts.index, iucn_counts.values, color="darkorange")   # ✏️ iucn_counts.index, iucn_counts.values

ax.set_title("IUCN Red List Category — Negros mammal records")
ax.set_xlabel("Category")
ax.set_ylabel("Number of records")

**🖊️ Look up** some of the specific `species` behind the EN/CR/VU records on Negros (`negros_mammals[negros_mammals["iucnRedListCategory"].isin(["EN", "CR", "VU"])]["species"].unique()`). What species are they? What threatens them?

In [ ]:
negros_mammals[negros_mammals["iucnRedListCategory"].isin(["EN", "CR", "VU"])]["species"].unique()   # ✏️ boolean filter (.isin) + column selection, same idea as before

> 💬 **Final discussion:** The **[IUCN Red List](https://www.iucnredlist.org/)** categorises extinction risk from least to most severe: **LC** (Least Concern) → **NT** (Near Threatened) → **VU** (Vulnerable) → **EN** (Endangered) → **CR** (Critically Endangered), plus **DD** (Data Deficient — not enough information to assess).
> - Most records are LC, which makes sense — common species get recorded more often. But which categories appear at all for Negros, and what does that tell you about the mammals sharing the island?
> - What vulnerable, endangered and critically endangered species did we find in this dataset? What do you think are the causes for their conservation status?
> - Keep this in mind for Parts III-IV: we're about to map land use and its change over time from satellite imagery!

---

### Your own biodiversity map

At the very start of Part II you built a tree-cover map — but only for Negros. We actually have that same Hansen dataset for the **whole Philippines** (`forest_cover_philippines_1km.nc`), and this section has had you collecting species records for the whole country too (`mammals`: 289 species, 48,640 records).

So: can we turn "a big pile of species sightings" into a **map** — one number per location, the same way `treecover2000` gives one number (% tree cover) per location? That number would be a simple measure of **biodiversity**: how many *different* species have been recorded near each spot.

Two new, short pandas tricks get us there: turning scattered points into a **grid**, and counting **unique** values per grid cell.

**💡 Trick #1 — rounding turns points into a grid.** Two GPS coordinates are almost never *exactly* equal, even for animals spotted right next to each other — one extra decimal of precision and they look like different locations. But if we **round** every coordinate to the same number of decimal places, nearby points collapse onto the same rounded value. That shared rounded value becomes a grid cell.

In [ ]:
# =============================================================
# 💡 EXAMPLE — rounding coordinates builds a grid
# =============================================================
toy = pd.DataFrame({
    "species": ["eagle", "eagle", "flying_fox", "tarsier", "flying_fox"],
    "lon":     [122.92,  122.93,  122.94,        124.60,    124.62],
    "lat":     [10.68,   10.67,   10.66,         9.70,      9.71],
})

toy["lon_grid"] = toy["lon"].round(1)
toy["lat_grid"] = toy["lat"].round(1)
toy

Notice rows 0-2 all round to the same `(lon_grid, lat_grid)` cell, `(122.9, 10.7)` — even though no two raw coordinates were identical. Rows 3-4 round to a different shared cell, `(124.6, 9.7)`. Rounding to 1 decimal place groups points within roughly ~10 km of each other into the same cell.

**🖊️ Your turn:** apply the same trick to the real `mammals` table.

First, a data-quality step: a handful of records sit way outside the Philippines (remember `negros_mammals["decimalLongitude"].describe()` showing points outside Negros — some are outside the *country*). Keep only records inside a generous Philippines bounding box before gridding, using `.between()` — same boolean-filtering idea as `df[condition]`, just checking a range instead of one value:

- Philippines bounding box: longitude 116–127°E, latitude 4.5–21.5°N
- Round `decimalLongitude`/`decimalLatitude` to 1 decimal place, same as the toy example, into new columns `lon_grid` / `lat_grid`

In [ ]:
in_ph = mammals["decimalLongitude"].between(116, 127) & mammals["decimalLatitude"].between(4.5, 21.5)   # ✏️ same boolean-filtering idea as before, with .between() for a range
mammals_ph = mammals[in_ph].copy()
print(f"{len(mammals)} records -> {len(mammals_ph)} after dropping out-of-bounds points")

mammals_ph["lon_grid"] = mammals_ph["decimalLongitude"].round(1)   # ✏️ same rounding trick as the toy example above
mammals_ph["lat_grid"] = mammals_ph["decimalLatitude"].round(1)    # ✏️ same rounding trick as the toy example above
mammals_ph.head()

**💡 Trick #2 — counting *unique* species per grid cell.**

You've already used `.value_counts()` to count *records*. What we want now is subtly different: per grid cell, how many *different* species were seen — a cell with 3 records of the same species shouldn't score higher than a cell with 2 records of 2 different species.

`.pivot_table()` builds exactly this kind of grid in one call: a row grouping, a column grouping, a value to summarize, and how to summarize it — here, `"nunique"`, short for *number of unique values*:

In [ ]:
# =============================================================
# 💡 EXAMPLE — pivot_table: row grouping x column grouping -> one summary value
# =============================================================
toy_richness = toy.pivot_table(index="lat_grid", columns="lon_grid",
                                values="species", aggfunc="nunique")
toy_richness

Both cells come out to **2** — cell `(10.7, 122.9)` had 3 records but only 2 distinct species (eagle, eagle, flying_fox); cell `(9.7, 124.6)` had 2 records and 2 distinct species (tarsier, flying_fox). `pivot_table` + `"nunique"` counted *species*, not *sightings*.

**🖊️ Your turn:** build the real richness grid — same call, real data.

- Same `pivot_table()` shape as above: `index="lat_grid"`, `columns="lon_grid"`, `values="species"`, `aggfunc="nunique"`
- Apply it to `mammals_ph`, and call the result `richness_ph`

In [ ]:
richness_ph = mammals_ph.pivot_table(index="lat_grid", columns="lon_grid",   # ✏️ same shape as toy_richness above, applied to mammals_ph
                                      values="species", aggfunc="nunique")
print(f"Grid shape: {richness_ph.shape}, richest cell: {int(richness_ph.max().max())} species")
richness_ph

**💡 One more ingredient — the same forest-cover data, for the whole country.**

You already know how to load and coarsen Hansen tree-cover data — you did exactly this for Negros, at the very start of Part II. `forest_cover_philippines_1km.nc` is the same dataset and variable, just for the whole archipelago. Two differences to watch for: its coordinates are named `lat`/`lon` here instead of `latitude`/`longitude`, and its native pixel size is different, so the coarsening factor needs adjusting to land on roughly the same ~0.1° cells as `richness_ph` (native pixels are ~0.0083°, so ~12 of them make up ~0.1°).

Load it, coarsen it, then we'll put both maps side by side.

In [ ]:
treecover_ph_ds = xr.open_dataset("../data/processed/forest_cover_philippines_1km.nc")
treecover_ph = treecover_ph_ds["treecover2000"]

treecover_ph_coarse = treecover_ph.coarsen(lat=12, lon=12, boundary="trim").mean()   # ✏️ same .coarsen() call as treecover_coarse earlier, adjusted for lat/lon names + factor
treecover_ph_coarse

In [ ]:
# --- Two-panel comparison: species richness vs. tree cover, whole Philippines ---
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

# 💡 given — pivot_table's output isn't an xarray DataArray, so we plot it with pcolormesh instead of .plot()
mesh = axes[0].pcolormesh(richness_ph.columns.values, richness_ph.index.values, richness_ph.values,
                           cmap="viridis", shading="nearest")
fig.colorbar(mesh, ax=axes[0], label="Unique species recorded", shrink=0.7)
axes[0].set_title("Mammal species richness (GBIF)")

treecover_ph_coarse.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100,
                         cbar_kwargs={"label": "Tree cover 2000 (%)"})   # ✏️ same .plot() call as treecover_coarse, right at the start of Part II
axes[1].set_title("Tree cover (Hansen)")

for ax in axes:
    ax.set_xlim(116, 127)
    ax.set_ylim(4.5, 21.5)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Where does the richness map light up brightest — does it track the tree-cover map, or does it look more like "wherever a biologist happened to visit"? Compare the hotspots to the sampling-bias pattern you already noticed in the eagle/flying-fox maps earlier in this Part. A high-richness cell can mean real biodiversity, or it can just mean *someone was there recording things* — with presence-only data like this, the map alone can't always tell you which.

---

## Well done — Part II complete!

You just:
- Connected yesterday's climate classification to actual vegetation and forest cover data
- Learned to build true-colour images from raw satellite bands (`np.dstack`)
- Compared satellite imagery across time, by eye
- Worked with real, messy biodiversity occurrence data (GBIF): filtering, null-checking, and conservation-status reporting
- Built your own species-richness grid from scratch (`round` + `pivot_table`) and used it to ask whether biodiversity tracks forest cover across the whole country

Next: **Part III**, where we go from satellite imagery to an actual land use classification.

## *PART III: From satellite imagery to land use*

Köppen designed his climate thresholds around vegetation zones; now let's flip that around and classify *land cover itself* directly from the Sentinel-2 imagery you already loaded in Part II — using the exact same recipe as Part I's homemade climate classifier: turn a decision rule into a `def` function, apply it to a whole grid at once, and map the result.

### 1) NDVI: turning two bands into one meaningful number

**Healthy, lush-green vegetation absorbs a lot of red light but reflects a lot of near-infrared**. Remember that we had these two bands available in our Sentinel-2 imagery? We can combine them in the [Normalized Difference Vegetation Index NDVI](https://en.wikipedia.org/wiki/Normalized_difference_vegetation_index), the most widely used metric to derive vegetation status from remote sensing data. NDVI is computed as:

**NDVI = (NIR − Red) / (NIR + Red)**

NDVI ranges from -1 to +1: strongly **negative** for water, **near zero** for bare soil/urban surfaces, and **high** (often >0.3) for healthy vegetation. Let's write a small reusable function for it: